# Analysis: Bolt, Clevis, Lug 
According to Roloff / Matek 2017 and mechanicalc.com  
**Author**: Artjom Lukanowski  
**Date**: 12.06.2026  

### Input: Generic data

In [36]:
import  math


""" INTEGRATION CASES
 c1: Clearance-fit everywhere 
 c2: Press-fit clevis <-> bolt, clearance bar <-> bolt 
 c3: Clearance clevis <-> bolt, press-fit bar <-> bolt """

int_case = "c1"

""" LOAD CASES
 stat: static
 dyn_pul: pulsating
 dyn_al: alternating """

ld_case = "stat"

""" BOLT HEAT TREATMENT 
ht: heat treatment
no_ht: no heat treatment """

bt_ht ="ht"

# MATERIAL PAIRING

mat_pair = "ST"

# FORCE
F_stat= 1.5*15000  #[N]

# FACTORS
#Clamping factor. Depending on integration case
if int_case == "c1": 
    k=1.6
else: k = 1.1

K_A = 1.5   #Schock factor


### Input: Geometry & Material
<img src="figures/GMA simplified.png" alt="GMA cross section" width="400"> <img src="figures/Lug Clevis.png" alt="GMA cross section" width="400"> 

In [49]:
# GEOMETRY
# Put "0" if value unknown for given parameters
unit = {
"length":"mm", "area":"mm²", "section mod":"mm³", "force":"N", "moment":"Nm", "stress": "MPa"}

geom = {
"bolt":     {"d": 0.6235*25.4,}, 
"lug":      {"d": 0.625*25.4,   "t": 0.567*25.4,    "a": 8.92,  "c": 8.92},
"clevis":   {"d": 0.625*25.4,   "t": 7.5,           "a": 12,    "c": 17}}

mat = {
"bolt":     {"Re":0,     "Rm":6.89476*(160 + 190) / 2,   "tau_ult":6.89476*95},
"lug":      {"Re":1000,  "Rm":1068,                     "tau_ult":0},
"clevis":   {"Re":1000,  "Rm":1068,                     "tau_ult":0}}


### BOLT Bending Moment & Allowable bending failure stress

In [48]:
#Bending moment
if int_case == "c1":
    Mb_max = 1/8 * F_stat * (geom["lug"]["t"]+ 2*geom["clevis"]["t"])
elif int_case == "c2":
    Mb_max = 1/8 * F_stat * geom["lug"]["t"]
elif int_case == "c3":
    Mb_max = 1/4 * F_stat * geom["clevis"]["t"]
else:
    raise ValueError("Invalid integration case")

basis = mat["bolt"]["Rm"] if bt_ht == "ht" else 400

#Bending stress

W = math.pi * geom["bolt"]["d"]**3 / 32
sigma_b = (K_A * Mb_max) / W

#Allowable bending failure stress"
if ld_case == "stat":
    sigma_b_lim = 0.3 * basis
elif ld_case == "dyn_pul":
    sigma_b_lim = 0.2 * basis
elif ld_case == "dyn_al":
    sigma_b_lim = 0.15 * basis

### BOLT Shear Strength

In [ ]:
A = math.pi *geom["bolt"]["d"]**2 / 4

# Shear bolt
tau_bt = (4/3) * (K_A * F_stat) / A / 2

if ld_case == "stat" and mat["bolt"]["tau_ult"] != 0:
    tau_bt_lim = mat["bolt"]["tau_ult"]
elif ld_case == "stat" and mat["bolt"]["tau_ult"] == 0:
    tau_bt_lim = 0.2 * mat["bolt"]["Rm"]
elif ld_case == "dyn_pul":
    tau_bt_lim = 0.15 * mat["bolt"]["Rm"]
elif ld_case == "dyn_al":
    tau_bt_lim = 0.1 * mat["bolt"]["Rm"]
else:
    tau_bt_lim = 400

tau_bt = 114.2 MPa
tau_bt_lim = 655.0 MPa


### LUG & CLEVIS Bearing Failure
![alt text](<figures/Lug Clevis bearing failure.png>)

In [ ]:
# Lug Bearing Failure
# Projected areas under hypothesis that only 115° out of 360° of entire perimeter is in contact under load

A_lg_br_proj = geom["bolt"]["d"] * geom["lug"]["t"] * 115 / 360
A_cl_br_proj = geom["bolt"]["d"] * geom["clevis"]["t"] * 115 / 360

if A_lg_br_proj >= A_cl_br_proj:
    p_bt_bear= K_A * 0.5 * F_stat / A_cl_br_proj 
else: 
    p_bt_bear= K_A * F_stat / A_lg_br_proj 

p_lg_bear= K_A * F_stat / A_lg_br_proj 
p_cl_bear= K_A * 0.5 * F_stat / A_cl_br_proj 

#Allowable bearing failure value
p_bt_bear_lim = 1.5 * mat["bolt"]["Rm"]
p_lg_bear_lim = 1.5 * mat["lug"]["Rm"]
p_cl_bear_lim = 1.5 * mat["clevis"]["Rm"]

### Surface pressure

In [ ]:
# Projected areas
A_lg_pr_proj = 2 * geom["lug"]["d"] * geom["lug"]["t"]
A_cl_pr_proj = 2 * geom["clevis"]["d"] * geom["clevis"]["t"]

457.2571499999999
238.125


#### Surface pressure bolt

In [42]:
# Bolt pressure
p_bt = K_A * F_stat / A_lg_pr_proj 

if ld_case == "stat":
    p_bt_lim = 0.35 * mat["bolt"]["Rm"]
elif ld_case == "dyn_pul":
    p_bt_lim = 0.25 * mat["bolt"]["Rm"]
elif ld_case == "dyn_al":
    p_bt_lim = 0.15 * mat["bolt"]["Rm"]
else:
    p_bt_lim = 400


#### Surface pressure lug

In [43]:
# Lug pressure
p_lg = K_A * F_stat / A_lg_pr_proj

if ld_case == "stat":
   p_lg_lim = 0.35 * mat["lug"]["Rm"]
elif ld_case == "dyn_pul":
    p_lg_lim = 0.25 * mat["lug"]["Rm"]
elif ld_case == "dyn_al":
    p_lg_lim = 0.15 * mat["lug"]["Rm"]
else:
    p_lg_lim = 400

#### Surface pressure clevis

In [44]:
p_cl = K_A * F_stat/2 / A_cl_pr_proj

if ld_case == "stat":
    p_cl_lim = 0.35 * mat["clevis"]["Rm"]
elif ld_case == "dyn_pul":
    p_cl_lim = 0.25 * mat["clevis"]["Rm"]
elif ld_case == "dyn_al":
    p_cl_lim = 0.15 * mat["clevis"]["Rm"]
else:
    p_cl_lim = 400


### Tension Failure Across Net Section (Flank)

In [45]:
sig_lg_flank = ((K_A * F_stat) / (2 * geom["lug"]["c"] * geom["lug"]["t"])) * (1 + (3/2)*(geom["bolt"]["d"] / geom["lug"]["c"] ))
sig_cl_flank = ((K_A * F_stat * 0.5) / (2 * geom["clevis"]["c"] * geom["clevis"]["t"])) * (1 + (3/2)*(geom["bolt"]["d"] / geom["clevis"]["c"]))

# Limits
if ld_case == "stat":
    sig_lg_flank_lim = 0.5 * mat["lug"]["Re"] if mat == "ST" else 0.5 * mat["lug"]["Rm"]
    sig_cl_flank_lim = 0.5 * mat["clevis"]["Re"] if mat == "ST" else 0.5 * mat["clevis"]["Rm"]
else:
    sig_lg_flank_lim = 0.2 * mat["lug"]["Re"] if mat == "ST" else 0.2 * mat["lug"]["Rm"]
    sig_cl_flank_lim = 0.2 * mat["clevis"]["Re"] if mat == "ST" else 0.2 * mat["clevis"]["Rm"]


### SUMMARY

In [46]:
print("\n=== RESULTS ===")
print(f"{'Name':<34} {'Parameter':<11} {'Value [MPa]':>13} {'Limit [MPa]':>15} {'Safety':>8} {'Check':>7}")
print("=" * 100)

def check_line(name, parameter, value, limit, safety):
    result = "OK" if value <= limit else "NOT OK"
    symbol = "✔" if result == "OK" else "✘"
    print(f"{name:<34} {parameter:<11} {value:>9.1f} {limit:>13.1f} {safety:>11.1f} {symbol:>8}")

check_line("Bending stress bolt", "sigma_b",sigma_b, sigma_b_lim, sigma_b_lim /sigma_b)
check_line("Shear stress bolt","tau_bt",tau_bt, tau_bt_lim, tau_bt_lim / tau_bt)
print("-" * 100)
check_line("Bearing failure bolt","p_bt_bear",p_bt_bear, p_bt_bear_lim, p_bt_bear_lim / p_bt_bear,)
check_line("Bearing failure lug","p_lg_bear",p_lg_bear, p_lg_bear_lim, p_lg_bear_lim / p_lg_bear,)
check_line("Bearing failure clevis","p_cl_bear",p_cl_bear, p_cl_bear_lim,  p_cl_bear_lim / p_cl_bear)
print("-" * 100)
check_line("Surface pressure bolt","p_bt",p_bt, p_bt_lim, p_bt_lim / p_bt)
check_line("Surface pressure lug","p_lg",p_lg, p_lg_lim, p_lg_lim / p_lg)
check_line("Surface pressure clevis","p_cl",p_cl, p_cl_lim, p_cl_lim / p_cl)
print("-" * 100)
check_line("Tension failure of lug flank","sig_lg_flank",sig_lg_flank, sig_lg_flank_lim, sig_lg_flank_lim / sig_lg_flank)
check_line("Tension failure of clevis flank","sig_cl_flank",sig_cl_flank, sig_cl_flank_lim, sig_cl_flank_lim / sig_cl_flank)


=== RESULTS ===
Name                               Parameter     Value [MPa]     Limit [MPa]   Safety   Check
Bending stress bolt                sigma_b         318.1         362.0         1.1        ✔
Shear stress bolt                  tau_bt          114.2         655.0         5.7        ✔
----------------------------------------------------------------------------------------------------
Bearing failure bolt               p_bt_bear       444.8        1809.9         4.1        ✔
Bearing failure lug                p_lg_bear       463.2        1602.0         3.5        ✔
Bearing failure clevis             p_cl_bear       444.8        1602.0         3.6        ✔
----------------------------------------------------------------------------------------------------
Surface pressure bolt              p_bt             73.8         422.3         5.7        ✔
Surface pressure lug               p_lg             73.8         373.8         5.1        ✔
Surface pressure clevis            p_cl    